# Wongnai QA System

ระบบถาม-ตอบอัจฉริยะสำหรับค้นหาและแนะนำร้านอาหาร โดยใช้ข้อมูลรีวิวจาก Wongnai ผ่านเทคนิค Semantic Search + Generative QA

## ความสามารถหลัก
- **Semantic Search**: ค้นหารีวิวที่เกี่ยวข้องด้วย Sentence Embedding + FAISS
- **Generative QA**: สร้างคำตอบสรุปจากรีวิวที่ค้นมาได้
- **Baseline vs Fine-tuned**: เปรียบเทียบผลลัพธ์ระหว่างโมเดลพื้นฐานกับโมเดลที่ fine-tune แล้ว
- **Web UI**: อินเทอร์เฟซสวยงามด้วย Gradio

## โครงสร้าง Notebook
1. Setup and Configuration
2. Data Preprocessing
3. Retrieval Module (FAISS)
4. Fine-tuning Module
5. QA Generation Module
6. Evaluation Module
7. Gradio Web UI
8. Pipeline Runner

## 1. Setup and Configuration

ติดตั้ง dependencies และกำหนดค่าคอนฟิก

In [1]:
# Install dependencies
!pip install torch>=2.0.0 transformers>=4.30.0 sentence-transformers>=2.2.0 faiss-cpu>=1.7.0 gradio>=4.0.0 pandas>=2.0.0 numpy>=1.24.0 scikit-learn>=1.3.0 pythainlp>=4.0.0 tqdm accelerate bitsandbytes peft

In [2]:
"""Configuration module for Wongnai QA System."""

import os
import torch

# Data directories and file paths
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "Dataset")
REVIEW_TRAIN_FILE = os.path.join(DATA_DIR, "review_dataset", "w_review_train.csv")
REVIEW_TEST_FILE = os.path.join(DATA_DIR, "review_dataset", "test_file.csv")
FOOD_DICT_FILE = os.path.join(DATA_DIR, "food_dictionary.txt")
QUERY_JUDGES_FILE = os.path.join(DATA_DIR, "labeled_queries_by_judges.txt")
QUERY_ALGO_FILE = os.path.join(DATA_DIR, "labeled_queries_by_algo.txt")

# Model configurations
MODEL_CONFIG = {
    "embedding_model": "intfloat/multilingual-e5-base",
    "qa_model": "scb10x/llama3.1-typhoon2-8b-instruct",
}

# FAISS and processed data paths
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
MODELS_DIR = os.path.join(BASE_DIR, "models")
FAISS_INDEX_PATH = os.path.join(MODELS_DIR, "faiss_index")
FINETUNED_FAISS_INDEX_PATH = os.path.join(MODELS_DIR, "finetuned_faiss_index")
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, "data", "processed")

# Retrieval settings
TOP_K = 5

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")

Using device: cpu
Data directory: e:\Master_Degree\NLP\Wongnai-QA-System\Dataset
Models directory: e:\Master_Degree\NLP\Wongnai-QA-System\models


## 2. Data Preprocessing

โมดูลสำหรับโหลด ทำความสะอาด และสกัด metadata จากข้อมูลรีวิว

In [3]:
"""Data preprocessing module for Wongnai QA System."""

import os
import re
import pandas as pd
from tqdm import tqdm
from typing import List, Dict, Set

# Keyword dictionaries for metadata extraction
CUISINE_KEYWORDS = {
    'thai': ['อาหารไทย', 'thai', 'ไทย'],
    'chinese': ['อาหารจีน', 'chinese', 'จีน'],
    'japanese': ['อาหารญี่ปุ่น', 'japanese', 'ญี่ปุ่น', 'เทปันยากิ'],
    'korean': ['อาหารเกาหลี', 'korean', 'เกาหลี'],
    'indian': ['อาหารอินเดีย', 'indian', 'อินเดีย'],
    'italian': ['อาหารอิตาลี', 'italian', 'อิตาลี'],
    'french': ['อาหารฝรั่งเศส', 'french', 'ฝรั่งเศส'],
    'vietnamese': ['อาหารเวียดนาม', 'vietnamese', 'เวียดนาม'],
    'mexican': ['mexican', 'เม็กซิกัน', 'เม็กซิโก'],
    'western': ['western', 'ฝรั่ง', 'อาหารฝรั่ง'],
    'fusion': ['ฟิวชั่น', 'fusion'],
    'isan': ['อีสาน', 'อิสาน', 'isan'],
    'southern': ['ใต้', 'southern thai', 'ภาคใต้'],
    'northern': ['เหนือ', 'northern thai', 'ภาคเหนือ'],
}

FOOD_TYPE_KEYWORDS = {
    'seafood': ['อาหารทะเล', 'ซีฟู้ด', 'seafood', 'ทะเล'],
    'pizza': ['พิซซ่า', 'pizza', 'พิซซา'],
    'bakery': ['เบเกอรี่', 'bakery', 'bakeries'],
    'dessert': ['ขนม', 'dessert', 'ของหวาน', 'ขนมหวาน'],
    'beverage': ['เครื่องดื่ม', 'beverage', 'drink'],
    'ice_cream': ['ไอศกรีม', 'ice cream', 'icecream'],
    'rice_curry': ['ข้าวแกง', 'แกงถุง', 'ข้าวราดแกง'],
    'noodle': ['ก๋วยเตี๋ยว', 'noodle', 'ก๋วยเตี๋ยว', 'เส้น', 'ขนมจีน'],
    'a_la_carte': ['ตามสั่ง', 'อาหารตามสั่ง'],
    'healthy': ['สุขภาพ', 'healthy', 'เพื่อสุขภาพ', 'คลีน', 'clean food'],
    'buffet': ['บุฟเฟ่ต์', 'buffet', 'ปิ้งย่าง', 'หมูกระทะ'],
    'shabu': ['ชาบู', 'shabu', 'ชาบูชาบู'],
    'steak': ['สเต็ก', 'steak'],
    'coffee': ['กาแฟ', 'coffee'],
    'tea': ['ชา', 'tea'],
    'bbq': ['ปิ้งย่าง', 'BBQ', 'barbecue', 'บาร์บีคิว'],
    'sushi': ['ซูชิ', 'sushi'],
    'ramen': ['ราเมน', 'ramen'],
    'dim_sum': ['ติ่มซำ', 'dim sum', 'dimsum'],
    'grill': ['หมูกระทะ', 'ย่าง', 'grill'],
    'salad': ['สลัด', 'salad'],
    'soup': ['ซุป', 'soup', 'น้ำซุป'],
}

ATMOSPHERE_KEYWORDS = {
    'luxury': ['หรูหรา', 'luxury', 'luxurious', 'หรู', 'hi-so'],
    'air_con': ['ติดแอร์', 'แอร์', 'air-con', 'aircon', 'air con'],
    'open_air': ['ร้านเปิด', 'open air', 'open-air', 'outdoor'],
    'street_food': ['ร้านข้างทาง', 'street food', 'ข้างทาง', 'แผงลอย'],
    'good_atmosphere': ['บรรยากาศดี', 'atmosphere', 'บรรยากาศ'],
    'good_view': ['วิวสวย', 'view', 'ทิวทัศน์'],
    'waterfront': ['ริมน้ำ', 'ริมคลอง', 'ริมแม่น้ำ', 'waterfront'],
    'beachfront': ['ริมทะเล', 'beachfront', 'beach', 'ทะเล'],
    'quiet': ['สงบ', 'quiet', 'เงียบ', 'ส่วนตัว', 'private'],
    'romantic': ['โรแมนติก', 'romantic', 'เดท', 'date'],
    'cafe': ['คาเฟ่', 'cafe', 'café'],
    'instagrammable': ['instagrammable', 'ig', 'ไอจี', 'ถ่ายรูปสวย', 'รูปสวย'],
    'cozy': ['น่านั่ง', 'cozy', 'comfortable', 'สบาย'],
    'spacious': ['กว้างขวาง', 'spacious', 'กว้าง', 'ใหญ่'],
    'family': ['ครอบครัว', 'family', 'เด็ก', 'kids', 'กลุ่มใหญ่'],
}

PRICE_KEYWORDS = {
    'expensive': ['ราคาแพง', 'แพง', 'expensive', 'pricey', 'high price'],
    'cheap': ['ราคาย่อมเยา', 'ถูก', 'ราคาถูก', 'cheap', 'inexpensive', 'ราคานักเรียน'],
    'worth': ['คุ้มค่า', 'คุ้ม', 'worth', 'value for money'],
    'reasonable': ['ราคาเหมาะสม', 'reasonable', 'fair price', 'เหมาะสม'],
    'premium': ['premium', 'พรีเมี่ยม', 'พรีเมียม'],
    'budget': ['ไม่แพง', 'budget', 'ราคาประหยัด', 'ประหยัด'],
}

LOCATION_KEYWORDS = {
    'beachside': ['ติดทะเล', 'ริมทะเล', 'beachside'],
    'mountain': ['บนเขา', 'mountain', 'ภูเขา', 'เขา'],
    'downtown': ['ใจกลางเมือง', 'downtown', 'city center', 'cbd'],
    'suburb': ['ชานเมือง', 'suburb', 'รอบนอก'],
    'mall': ['ห้างสรรพสินค้า', 'mall', 'shopping mall', 'ห้าง', 'department store'],
    'market': ['ตลาดนัด', 'market', 'ตลาด'],
}

PROVINCE_NAMES = {
    'bangkok': ['กรุงเทพ', 'bangkok', 'กรุงเทพมหานคร', 'bkk'],
    'chiang_mai': ['เชียงใหม่', 'chiang mai', 'chiangmai'],
    'pattaya': ['พัทยา', 'pattaya'],
    'phuket': ['ภูเก็ต', 'phuket'],
    'hua_hin': ['หัวหิน', 'hua hin', 'huahin'],
    'khao_yai': ['เขาใหญ่', 'khao yai', 'khaoyai'],
    'ayutthaya': ['อยุธยา', 'ayutthaya'],
    'kanchanaburi': ['กาญจนบุรี', 'kanchanaburi'],
    'rayong': ['ระยอง', 'rayong'],
    'chonburi': ['ชลบุรี', 'chonburi'],
    'korat': ['นครราชสีมา', 'โคราช', 'korat', 'nakhon ratchasima'],
    'khon_kaen': ['ขอนแก่น', 'khon kaen', 'khonkaen'],
    'songkhla': ['สงขลา', 'songkhla', 'หาดใหญ่', 'hat yai', 'hatyai'],
    'chiang_rai': ['เชียงราย', 'chiang rai', 'chiangrai'],
    'surat': ['สุราษฎร์ธานี', 'surat thani'],
    'samui': ['สมุย', 'samui', 'koh samui', 'เกาะสมุย'],
    'krabi': ['กระบี่', 'krabi'],
    'trat': ['ตราด', 'trat'],
    'chantaburi': ['จันทบุรี', 'chantaburi', 'chanthaburi'],
    'nakhon_pathom': ['นครปฐม', 'nakhon pathom'],
    'ratchaburi': ['ราชบุรี', 'ratchaburi'],
    'lampang': ['ลำปาง', 'lampang'],
    'nan': ['น่าน', 'nan'],
    'mae_hong_son': ['แม่ฮ่องสอน', 'mae hong son', 'maehongson'],
    'sukhothai': ['สุโขทัย', 'sukhothai'],
    'phitsanulok': ['พิษณุโลก', 'phitsanulok'],
    'udon_thani': ['อุดรธานี', 'udon thani', 'udonthani'],
    'sakon_nakhon': ['สกลนคร', 'sakon nakhon', 'sakonnakhon'],
    'nakhon_nayok': ['นครนายก', 'nakhon nayok'],
    'prachuap': ['ประจวบคีรีขันธ์', 'prachuap khiri khan', 'prachuapkhirikhan', 'หัวหิน'],
}


def load_reviews(file_path: str) -> pd.DataFrame:
    """Load review data from CSV file."""
    df = pd.read_csv(file_path, sep=';', header=None, names=['review_text', 'star_rating'])
    df = df.dropna(subset=['review_text'])
    df['star_rating'] = df['star_rating'].astype(int)
    return df


def load_food_dictionary(file_path: str) -> List[str]:
    """Load food dictionary from text file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        foods = [line.strip() for line in f if line.strip()]
    return foods


def load_queries(file_path: str) -> List[str]:
    """Load query data from text file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                query = line.replace('|', ' ')
                query = re.sub(r'\s+', ' ', query).strip()
                queries.append(query)
    return queries


def clean_review(text: str) -> str:
    """Clean and normalize review text."""
    if pd.isna(text):
        return ""
    text = re.sub(r'\s+', ' ', str(text))
    text = text.strip()
    if len(text) < 20:
        return ""
    return text


def _detect_keywords(text: str, keyword_dict: Dict[str, List[str]]) -> List[str]:
    """Helper function to detect keywords from text."""
    text_lower = text.lower()
    matched = []
    for category, keywords in keyword_dict.items():
        if any(kw in text_lower for kw in keywords):
            matched.append(category)
    return matched


def extract_metadata(review_text: str, food_dict: Set[str]) -> Dict:
    """Extract structured metadata from review text."""
    if pd.isna(review_text):
        review_text = ""
    text = str(review_text)
    text_lower = text.lower()
    
    metadata = {
        'cuisine_type': _detect_keywords(text, CUISINE_KEYWORDS),
        'food_type': _detect_keywords(text, FOOD_TYPE_KEYWORDS),
        'atmosphere': _detect_keywords(text, ATMOSPHERE_KEYWORDS),
        'price_level': _detect_keywords(text, PRICE_KEYWORDS),
        'location': _detect_keywords(text, LOCATION_KEYWORDS),
    }
    
    detected_provinces = _detect_keywords(text, PROVINCE_NAMES)
    metadata['location'].extend(detected_provinces)
    metadata['location'] = list(dict.fromkeys(metadata['location']))
    
    mentioned_foods = []
    for food in food_dict:
        if food.lower() in text_lower:
            mentioned_foods.append(food)
            if len(mentioned_foods) >= 5:
                break
    metadata['mentioned_foods'] = mentioned_foods
    
    return metadata


def create_search_text(row: pd.Series) -> str:
    """Create a rich search text combining review and metadata."""
    cuisine = ', '.join(row.get('cuisine_type', [])) if isinstance(row.get('cuisine_type'), list) else ''
    food = ', '.join(row.get('food_type', [])) if isinstance(row.get('food_type'), list) else ''
    atmos = ', '.join(row.get('atmosphere', [])) if isinstance(row.get('atmosphere'), list) else ''
    price = ', '.join(row.get('price_level', [])) if isinstance(row.get('price_level'), list) else ''
    loc = ', '.join(row.get('location', [])) if isinstance(row.get('location'), list) else ''
    
    review = str(row.get('review_text', ''))[:500]
    
    parts = [p for p in [cuisine, food, atmos, price, loc] if p]
    metadata_str = ' '.join(parts)
    
    if metadata_str:
        return f"{metadata_str}: {review}"
    return review


def process_all_reviews(
    reviews_df: pd.DataFrame,
    food_dict_set: Set[str],
    max_reviews: int = 50000
) -> pd.DataFrame:
    """Process all reviews: clean, sample, extract metadata, and save."""
    print(f"Starting with {len(reviews_df)} reviews...")
    
    print("Cleaning reviews...")
    tqdm.pandas(desc="Cleaning")
    reviews_df['review_text'] = reviews_df['review_text'].progress_apply(clean_review)
    
    reviews_df = reviews_df[reviews_df['review_text'] != '']
    print(f"After cleaning: {len(reviews_df)} reviews")
    
    if len(reviews_df) > max_reviews:
        print(f"Sampling {max_reviews} reviews for processing...")
        reviews_df = reviews_df.sample(n=max_reviews, random_state=42)
    
    print("Extracting metadata...")
    tqdm.pandas(desc="Metadata extraction")
    metadata_series = reviews_df['review_text'].progress_apply(
        lambda x: extract_metadata(x, food_dict_set)
    )
    
    metadata_df = pd.DataFrame(metadata_series.tolist())
    reviews_df = pd.concat([reviews_df.reset_index(drop=True), metadata_df], axis=1)
    
    print("Creating search text...")
    reviews_df['search_text'] = reviews_df.apply(create_search_text, axis=1)
    
    os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
    output_path = os.path.join(PROCESSED_DATA_PATH, 'processed_reviews.pkl')
    reviews_df.to_pickle(output_path)
    print(f"Saved processed reviews to {output_path}")
    
    return reviews_df

print("Data preprocessing module loaded successfully!")

Data preprocessing module loaded successfully!


## 3. Retrieval Module (FAISS)

โมดูลสำหรับสร้าง FAISS index และค้นหาด้วย Semantic Search

In [4]:
"""Retrieval module for Wongnai QA System."""

import os
import pickle
from typing import List, Dict, Optional

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


class WongnaiRetriever:
    """Retriever class for semantic search over Wongnai restaurant reviews."""
    
    def __init__(
        self,
        model_name: Optional[str] = None,
        index_path: Optional[str] = None,
        is_finetuned: bool = False,
    ):
        """Initialize the retriever."""
        self.model_name = model_name or MODEL_CONFIG['embedding_model']
        self.model = SentenceTransformer(self.model_name, device=DEVICE)
        
        if index_path is None:
            if is_finetuned:
                self.index_path = FINETUNED_FAISS_INDEX_PATH
            else:
                self.index_path = FAISS_INDEX_PATH
        else:
            self.index_path = index_path
        
        self.df_path = self.index_path + "_df.pkl"
        self.is_e5_model = 'e5' in self.model_name.lower()
        
        self.index = None
        self.df = None
        self.embeddings = None
    
    def build_index(self, processed_df: pd.DataFrame, batch_size: int = 64) -> None:
        """Build the FAISS index from processed reviews."""
        self.df = processed_df.reset_index(drop=True)
        
        texts = self.df['search_text'].tolist()
        
        if self.is_e5_model:
            texts = [f"passage: {text}" for text in texts]
        
        print(f"Encoding {len(texts)} documents with {self.model_name}...")
        self.embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        
        faiss.normalize_L2(self.embeddings)
        
        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.embeddings)
        
        os.makedirs(os.path.dirname(self.index_path) or ".", exist_ok=True)
        faiss.write_index(self.index, self.index_path)
        self.df.to_pickle(self.df_path)
        
        print(f"Saved FAISS index to {self.index_path}")
        print(f"Saved DataFrame to {self.df_path}")
    
    def load_index(self) -> None:
        """Load the FAISS index and DataFrame from disk."""
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(f"FAISS index not found at {self.index_path}")
        if not os.path.exists(self.df_path):
            raise FileNotFoundError(f"DataFrame pickle not found at {self.df_path}")
        
        self.index = faiss.read_index(self.index_path)
        self.df = pd.read_pickle(self.df_path)
        
        print(f"Loaded FAISS index from {self.index_path}")
        print(f"Loaded DataFrame from {self.df_path}")
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for similar reviews."""
        if self.index is None or self.df is None:
            raise RuntimeError("Index not loaded. Call load_index() or build_index() first.")
        
        query_text = query
        if self.is_e5_model:
            query_text = f"query: {query}"
        
        query_embedding = self.model.encode(
            [query_text],
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        
        faiss.normalize_L2(query_embedding)
        scores, indices = self.index.search(query_embedding, top_k)
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            
            row = self.df.iloc[idx]
            result = {
                'review_text': row.get('review_text', ''),
                'star_rating': int(row.get('star_rating', 0)),
                'score': float(score),
                'cuisine_type': row.get('cuisine_type', []),
                'food_type': row.get('food_type', []),
                'atmosphere': row.get('atmosphere', []),
                'price_level': row.get('price_level', []),
                'location': row.get('location', []),
                'mentioned_foods': row.get('mentioned_foods', []),
            }
            results.append(result)
        
        return results
    
    def search_with_filters(
        self,
        query: str,
        top_k: int = 5,
        min_rating: Optional[int] = None,
        cuisine_filter: Optional[str] = None,
        food_type_filter: Optional[str] = None,
        location_filter: Optional[str] = None,
    ) -> List[Dict]:
        """Search with post-filtering on metadata."""
        initial_k = top_k * 3
        results = self.search(query, top_k=initial_k)
        
        filtered_results = []
        for result in results:
            if min_rating is not None and result['star_rating'] < min_rating:
                continue
            
            if cuisine_filter is not None:
                cuisines = [c.lower() for c in result['cuisine_type']]
                if cuisine_filter.lower() not in cuisines:
                    continue
            
            if food_type_filter is not None:
                food_types = [f.lower() for f in result['food_type']]
                if food_type_filter.lower() not in food_types:
                    continue
            
            if location_filter is not None:
                locations = [l.lower() for l in result['location']]
                if location_filter.lower() not in locations:
                    continue
            
            filtered_results.append(result)
        
        return filtered_results[:top_k]


def build_baseline_index(processed_df: pd.DataFrame) -> WongnaiRetriever:
    """Build a baseline index with the default embedding model."""
    retriever = WongnaiRetriever()
    retriever.build_index(processed_df)
    return retriever


def build_finetuned_index(
    processed_df: pd.DataFrame,
    finetuned_model_path: str,
) -> WongnaiRetriever:
    """Build an index with a finetuned model."""
    retriever = WongnaiRetriever(
        model_name=finetuned_model_path,
        index_path=FINETUNED_FAISS_INDEX_PATH,
        is_finetuned=True,
    )
    retriever.build_index(processed_df)
    return retriever

print("Retrieval module loaded successfully!")

e:\Master_Degree\NLP\Wongnai-QA-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retrieval module loaded successfully!


## 4. Fine-tuning Module

โมดูลสำหรับ Fine-tune Embedding Model ด้วย Contrastive Learning

In [5]:
"""Fine-tuning module for the embedding model in Wongnai QA System."""

import os
import random
from typing import List, Set, Dict

import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers import losses, evaluation
from torch.utils.data import DataLoader
from tqdm import tqdm


def generate_training_pairs(
    processed_df: pd.DataFrame,
    food_dict: List[str],
    queries_judges: List[str],
    queries_algo: List[str],
    num_pairs: int = 10000
) -> List[InputExample]:
    """Generate training pairs for contrastive learning."""
    training_pairs = []
    food_dict_set = set(food_dict)
    
    all_queries = list(set(queries_judges + queries_algo))
    print(f"Total unique queries: {len(all_queries)}")
    
    cuisine_to_indices: Dict[str, List[int]] = {}
    foodtype_to_indices: Dict[str, List[int]] = {}
    location_to_indices: Dict[str, List[int]] = {}
    high_rating_indices: List[int] = []
    
    for idx, row in tqdm(processed_df.iterrows(), total=len(processed_df), desc="Building lookup"):
        if row.get('star_rating', 0) >= 4:
            high_rating_indices.append(idx)
        
        for cuisine in row.get('cuisine_type', []):
            if cuisine not in cuisine_to_indices:
                cuisine_to_indices[cuisine] = []
            cuisine_to_indices[cuisine].append(idx)
        
        for food_type in row.get('food_type', []):
            if food_type not in foodtype_to_indices:
                foodtype_to_indices[food_type] = []
            foodtype_to_indices[food_type].append(idx)
        
        for loc in row.get('location', []):
            if loc not in location_to_indices:
                location_to_indices[loc] = []
            location_to_indices[loc].append(idx)
    
    print(f"High rating reviews: {len(high_rating_indices)}")
    print(f"Cuisine groups: {len(cuisine_to_indices)}")
    print(f"Food type groups: {len(foodtype_to_indices)}")
    print(f"Location groups: {len(location_to_indices)}")
    
    # Strategy A: Query-Review pairs (30%)
    num_query_pairs = int(num_pairs * 0.3)
    print(f"\nGenerating {num_query_pairs} query-review pairs...")
    
    review_texts = processed_df['review_text'].tolist()
    search_texts = processed_df['search_text'].tolist()
    
    query_pairs_count = 0
    max_attempts = num_query_pairs * 10
    attempts = 0
    
    while query_pairs_count < num_query_pairs and attempts < max_attempts:
        attempts += 1
        query = random.choice(all_queries)
        query_terms = query.lower().split()
        
        matching_indices = []
        for idx, review_text in enumerate(review_texts):
            review_lower = review_text.lower()
            if any(len(term) >= 3 and term in review_lower for term in query_terms):
                matching_indices.append(idx)
        
        if len(matching_indices) >= 1:
            pos_idx = random.choice(matching_indices)
            training_pairs.append(InputExample(
                texts=[query, search_texts[pos_idx]]
            ))
            query_pairs_count += 1
    
    print(f"Generated {query_pairs_count} query-review pairs")
    
    # Strategy B: Similar Review pairs (40%)
    num_similar_pairs = int(num_pairs * 0.4)
    print(f"\nGenerating {num_similar_pairs} similar review pairs...")
    
    all_groups = []
    for indices in cuisine_to_indices.values():
        if len(indices) >= 2:
            all_groups.append(('cuisine', indices))
    for indices in foodtype_to_indices.values():
        if len(indices) >= 2:
            all_groups.append(('food_type', indices))
    for indices in location_to_indices.values():
        if len(indices) >= 2:
            all_groups.append(('location', indices))
    
    print(f"Available metadata groups: {len(all_groups)}")
    
    similar_pairs_count = 0
    max_attempts = num_similar_pairs * 5
    attempts = 0
    
    while similar_pairs_count < num_similar_pairs and attempts < max_attempts:
        attempts += 1
        valid_groups = [g for g in all_groups if len(g[1]) >= 2]
        if not valid_groups:
            break
        
        group_type, indices = random.choice(valid_groups)
        idx1, idx2 = random.sample(indices, 2)
        
        training_pairs.append(InputExample(
            texts=[search_texts[idx1], search_texts[idx2]]
        ))
        similar_pairs_count += 1
    
    print(f"Generated {similar_pairs_count} similar review pairs")
    
    # Strategy C: Rating-based pairs (30%)
    num_rating_pairs = int(num_pairs * 0.3)
    print(f"\nGenerating {num_rating_pairs} rating-based pairs...")
    
    rating_pairs_count = 0
    max_attempts = num_rating_pairs * 5
    attempts = 0
    
    review_food_types = processed_df['food_type'].tolist()
    
    while rating_pairs_count < num_rating_pairs and attempts < max_attempts:
        attempts += 1
        
        if len(high_rating_indices) < 2:
            break
        
        idx1 = random.choice(high_rating_indices)
        idx2 = random.choice(high_rating_indices)
        
        if idx1 == idx2:
            continue
        
        foods1 = set(review_food_types[idx1]) if isinstance(review_food_types[idx1], list) else set()
        foods2 = set(review_food_types[idx2]) if isinstance(review_food_types[idx2], list) else set()
        
        if foods1 and foods2 and foods1.intersection(foods2):
            training_pairs.append(InputExample(
                texts=[search_texts[idx1], search_texts[idx2]]
            ))
            rating_pairs_count += 1
    
    print(f"Generated {rating_pairs_count} rating-based pairs")
    
    # Fill remaining pairs
    total_generated = len(training_pairs)
    if total_generated < num_pairs:
        print(f"\nFilling remaining {num_pairs - total_generated} pairs with random sampling...")
        while len(training_pairs) < num_pairs:
            idx1, idx2 = random.sample(range(len(processed_df)), 2)
            training_pairs.append(InputExample(
                texts=[search_texts[idx1], search_texts[idx2]]
            ))
    
    random.shuffle(training_pairs)
    training_pairs = training_pairs[:num_pairs]
    
    print(f"\nTotal training pairs: {len(training_pairs)}")
    return training_pairs


def finetune_model(
    training_pairs: List[InputExample],
    base_model_name: str = None,
    output_path: str = 'models/finetuned_embedding',
    epochs: int = 3,
    batch_size: int = 16,
    warmup_steps: int = 100
) -> str:
    """Fine-tune a sentence-transformers model with contrastive learning."""
    if base_model_name is None:
        base_model_name = MODEL_CONFIG['embedding_model']
    
    print(f"\nLoading base model: {base_model_name}")
    model = SentenceTransformer(base_model_name, device=DEVICE)
    
    train_dataloader = DataLoader(
        training_pairs,
        shuffle=True,
        batch_size=batch_size
    )
    
    train_loss = losses.MultipleNegativesRankingLoss(model)
    
    total_steps = len(train_dataloader) * epochs
    print(f"Training steps per epoch: {len(train_dataloader)}")
    print(f"Total training steps: {total_steps}")
    
    os.makedirs(output_path, exist_ok=True)
    
    print(f"\nStarting fine-tuning for {epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Warmup steps: {warmup_steps}")
    print(f"Output path: {output_path}")
    
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        output_path=output_path,
        show_progress_bar=True,
    )
    
    print(f"\nFine-tuning completed!")
    print(f"Model saved to: {output_path}")
    
    return output_path

print("Fine-tuning module loaded successfully!")

Fine-tuning module loaded successfully!


## 5. QA Generation Module

โมดูลสำหรับสร้างคำตอบจากรีวิวที่ค้นพบ (Template-based และ LLM-based)

In [6]:
"""Question answering generation module for Wongnai QA System."""

from typing import List, Dict, Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


class WongnaiQAGenerator:
    """Generator class for creating natural language answers from retrieved reviews."""
    
    def __init__(self, use_llm: bool = False, model_name: Optional[str] = None):
        """Initialize the QA generator."""
        self.use_llm = use_llm
        self.model = None
        self.tokenizer = None
        self.model_name = model_name or MODEL_CONFIG['qa_model']
        
        if self.use_llm:
            if not torch.cuda.is_available():
                print("Warning: CUDA not available. Falling back to template mode.")
                self.use_llm = False
                return
            
            try:
                print(f"Loading LLM model: {self.model_name}")
                
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                )
                
                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
                if self.tokenizer.pad_token is None:
                    self.tokenizer.pad_token = self.tokenizer.eos_token
                
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    torch_dtype=torch.float16,
                    trust_remote_code=True,
                )
                
                print(f"Successfully loaded LLM model on {DEVICE}")
                
            except Exception as e:
                print(f"Error loading LLM model: {e}")
                print("Falling back to template mode.")
                self.use_llm = False
                self.model = None
                self.tokenizer = None
    
    def _format_stars(self, rating: int) -> str:
        """Format star rating as star symbols."""
        return '*' * rating
    
    def generate_answer_template(self, query: str, results: List[Dict]) -> str:
        """Generate answer using template-based approach."""
        if not results:
            return 'ไม่พบข้อมูลที่ตรงกับคำถามของคุณ'
        
        answer_parts = [
            f'จากการค้นหา "{query}" พบ {len(results)} ร้านที่เกี่ยวข้อง:\n'
        ]
        
        for i, result in enumerate(results, 1):
            star_rating = result.get('star_rating', 0)
            score = result.get('score', 0.0)
            review_text = result.get('review_text', '')
            cuisine_type = result.get('cuisine_type', [])
            food_type = result.get('food_type', [])
            location = result.get('location', [])
            
            result_header = f'\nร้านที่ {i} (Rating: {star_rating}/5 {self._format_stars(star_rating)}, ความเกี่ยวข้อง: {score:.0%})'
            answer_parts.append(result_header)
            
            metadata_parts = []
            if cuisine_type:
                metadata_parts.append(f'ประเภทอาหาร: {", ".join(cuisine_type)}')
            if food_type:
                metadata_parts.append(f'แนวอาหาร: {", ".join(food_type)}')
            if location:
                metadata_parts.append(f'ที่ตั้ง: {", ".join(location)}')
            
            if metadata_parts:
                answer_parts.append(' | '.join(metadata_parts))
            
            excerpt = review_text[:300].strip()
            if len(review_text) > 300:
                excerpt += '...'
            answer_parts.append(f'รีวิว: {excerpt}')
            
            answer_parts.append('-' * 50)
        
        return '\n'.join(answer_parts)
    
    def generate_answer_llm(self, query: str, results: List[Dict]) -> str:
        """Generate answer using LLM-based approach."""
        if not results:
            return 'ไม่พบข้อมูลที่ตรงกับคำถามของคุณ'
        
        if self.model is None or self.tokenizer is None:
            return self.generate_answer_template(query, results)
        
        context_parts = []
        for i, result in enumerate(results, 1):
            star_rating = result.get('star_rating', 0)
            review_text = result.get('review_text', '')
            context_parts.append(f'รีวิว {i} (Rating: {star_rating}/5): {review_text[:400]}')
        
        context = '\n\n'.join(context_parts)
        
        system_prompt = (
            'คุณเป็นผู้เชี่ยวชาญแนะนำร้านอาหาร ตอบคำถามจากข้อมูลรีวิวที่ให้มา '
            'ตอบเป็นภาษาไทย กระชับ ให้ข้อมูลครบ ระบุ star rating ของแต่ละร้านด้วย'
        )
        
        user_prompt = f'คำถาม: {query}\n\nข้อมูลรีวิว:\n{context}'
        
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
        
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = self.tokenizer(
                prompt,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=2048,
            )
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            
            generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
            response = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
            
            return response.strip()
            
        except Exception as e:
            print(f"Error during LLM generation: {e}")
            return self.generate_answer_template(query, results)
    
    def generate_answer(self, query: str, results: List[Dict]) -> str:
        """Generate answer based on query and retrieved results."""
        if self.use_llm and self.model is not None:
            return self.generate_answer_llm(query, results)
        else:
            return self.generate_answer_template(query, results)
    
    def format_results_for_display(
        self,
        query: str,
        results: List[Dict],
        answer: str,
    ) -> Dict:
        """Format results for web UI display."""
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_result = {
                'rank': i,
                'review_text': result.get('review_text', ''),
                'star_rating': result.get('star_rating', 0),
                'star_display': self._format_stars(result.get('star_rating', 0)),
                'relevance_score': result.get('score', 0.0),
                'cuisine_type': result.get('cuisine_type', []),
                'food_type': result.get('food_type', []),
                'atmosphere': result.get('atmosphere', []),
                'price_level': result.get('price_level', []),
                'location': result.get('location', []),
                'mentioned_foods': result.get('mentioned_foods', []),
            }
            formatted_results.append(formatted_result)
        
        return {
            'query': query,
            'answer': answer,
            'num_results': len(results),
            'results': formatted_results,
        }

print("QA Generator module loaded successfully!")

QA Generator module loaded successfully!


## 6. Evaluation Module

โมดูลสำหรับประเมินผลระบบ QA

In [7]:
"""Evaluation module for Wongnai QA System."""

from typing import List, Dict, Optional
import pandas as pd
import numpy as np
from collections import defaultdict


def get_demo_queries() -> List[Dict]:
    """Return a list of demo queries covering 5 required categories."""
    queries = [
        {"category": "cuisine", "query": "อาหารญี่ปุ่นอร่อย", "description": "ค้นหาร้านอาหารญี่ปุ่นที่ได้รับความนิยม"},
        {"category": "cuisine", "query": "ร้านอาหารจีนแนะนำ", "description": "ค้นหาร้านอาหารจีนที่แนะนำ"},
        {"category": "cuisine", "query": "อาหารอิตาลีดี", "description": "ค้นหาร้านอาหารอิตาลีคุณภาพดี"},
        {"category": "cuisine", "query": "ร้านอาหารไทยรสเด็ด", "description": "ค้นหาร้านอาหารไทยรสชาติจัดจ้าน"},
        {"category": "cuisine", "query": "อาหารอินเดียเผ็ด", "description": "ค้นหาร้านอาหารอินเดียรสเผ็ด"},
        
        {"category": "food_type", "query": "อาหารทะเลสด", "description": "ค้นหาร้านอาหารทะเลสดใหม่"},
        {"category": "food_type", "query": "พิซซ่าอร่อย", "description": "ค้นหาร้านพิซซ่าอร่อย"},
        {"category": "food_type", "query": "ร้านเบเกอรี่ดี", "description": "ค้นหาร้านเบเกอรี่คุณภาพดี"},
        {"category": "food_type", "query": "ร้านกาแฟบรรยากาศดี", "description": "ค้นหาร้านกาแฟบรรยากาศดี"},
        {"category": "food_type", "query": "ก๋วยเตี๋ยวเส้นเล็ก", "description": "ค้นหาร้านก๋วยเตี๋ยวเส้นเล็กอร่อย"},
        
        {"category": "atmosphere_price", "query": "ร้านอาหารหรูหรา", "description": "ค้นหาร้านอาหารระดับหรู"},
        {"category": "atmosphere_price", "query": "ร้านข้างทางราคาถูก", "description": "ค้นหาร้านข้างทางราคาประหยัด"},
        {"category": "atmosphere_price", "query": "ร้านบรรยากาศดีติดแอร์", "description": "ค้นหาร้านอาหารบรรยากาศดีมีแอร์"},
        {"category": "atmosphere_price", "query": "คาเฟ่น่านั่ง", "description": "ค้นหาคาเฟ่บรรยากาศน่านั่ง"},
        {"category": "atmosphere_price", "query": "ร้านอาหารราคานักศึกษา", "description": "ค้นหาร้านอาหารราคาไม่แพง"},
        
        {"category": "location", "query": "ร้านอาหารเชียงใหม่", "description": "ค้นหาร้านอาหารในเชียงใหม่"},
        {"category": "location", "query": "ร้านติดทะเลพัทยา", "description": "ค้นหาร้านอาหารติดทะเลที่พัทยา"},
        {"category": "location", "query": "ร้านอาหารบนเขาใหญ่", "description": "ค้นหาร้านอาหารบรรยากาศเขาใหญ่"},
        {"category": "location", "query": "ร้านอาหารกรุงเทพ", "description": "ค้นหาร้านอาหารในกรุงเทพ"},
        {"category": "location", "query": "ร้านอาหารหัวหิน", "description": "ค้นหาร้านอาหารในหัวหิน"},
        
        {"category": "combined", "query": "อาหารทะเลแบบไทยติดชายหาดพัทยา", "description": "ค้นหาอาหารทะเลสไตล์ไทยริมหาดพัทยา"},
        {"category": "combined", "query": "ร้านอาหารญี่ปุ่นราคาไม่แพงเชียงใหม่", "description": "ค้นหาร้านอาหารญี่ปุ่นราคาประหยัดในเชียงใหม่"},
        {"category": "combined", "query": "คาเฟ่บรรยากาศดีถ่ายรูปสวยกรุงเทพ", "description": "ค้นหาคาเฟ่บรรยากาศดีถ่ายรูปสวยในกรุงเทพ"},
        {"category": "combined", "query": "ร้านก๋วยเตี๋ยวอร่อยราคาถูก", "description": "ค้นหาร้านก๋วยเตี๋ยวอร่อยราคาไม่แพง"},
        {"category": "combined", "query": "ร้านอาหารไทยบรรยากาศดีวิวสวย", "description": "ค้นหาร้านอาหารไทยบรรยากาศดีวิวสวย"},
        {"category": "combined", "query": "อาหารอิตาลีราคาหรูกรุงเทพ", "description": "ค้นหาร้านอาหารอิตาลีระดับหรูในกรุงเทพ"},
    ]
    
    return queries


def evaluate_retrieval(
    retriever: WongnaiRetriever,
    test_queries: List[str],
    ground_truth_labels: Optional[List[List[str]]] = None,
) -> Dict:
    """Evaluate retrieval performance on test queries."""
    if retriever.index is None or retriever.df is None:
        raise RuntimeError("Retriever index not loaded. Call load_index() first.")
    
    all_scores = []
    all_ratings = []
    all_metadata_coverage = []
    query_results = []
    
    for i, query in enumerate(test_queries):
        results = retriever.search(query, top_k=5)
        
        scores = [r['score'] for r in results]
        ratings = [r['star_rating'] for r in results]
        
        coverage_per_result = []
        for r in results:
            has_metadata = bool(
                r.get('cuisine_type', []) or 
                r.get('food_type', []) or 
                r.get('location', [])
            )
            coverage_per_result.append(1.0 if has_metadata else 0.0)
        
        avg_score = np.mean(scores) if scores else 0.0
        avg_rating = np.mean(ratings) if ratings else 0.0
        avg_coverage = np.mean(coverage_per_result) if coverage_per_result else 0.0
        
        all_scores.extend(scores)
        all_ratings.extend(ratings)
        all_metadata_coverage.extend(coverage_per_result)
        
        query_results.append({
            'query': query,
            'num_results': len(results),
            'avg_score': avg_score,
            'avg_rating': avg_rating,
            'metadata_coverage': avg_coverage,
            'results': results,
        })
    
    metrics = {
        'avg_retrieval_score': float(np.mean(all_scores)) if all_scores else 0.0,
        'avg_star_rating': float(np.mean(all_ratings)) if all_ratings else 0.0,
        'metadata_coverage': float(np.mean(all_metadata_coverage) * 100) if all_metadata_coverage else 0.0,
        'num_queries': len(test_queries),
        'total_results': len(all_scores),
        'query_results': query_results,
    }
    
    return metrics


def compare_retrievers(
    baseline_retriever: WongnaiRetriever,
    finetuned_retriever: WongnaiRetriever,
    test_queries: List[str],
) -> pd.DataFrame:
    """Compare baseline vs finetuned retriever performance."""
    baseline_metrics = evaluate_retrieval(baseline_retriever, test_queries)
    finetuned_metrics = evaluate_retrieval(finetuned_retriever, test_queries)
    
    comparison_data = []
    for i, query in enumerate(test_queries):
        baseline_result = baseline_metrics['query_results'][i]
        finetuned_result = finetuned_metrics['query_results'][i]
        
        baseline_score = baseline_result['avg_score']
        finetuned_score = finetuned_result['avg_score']
        
        if baseline_score > 0:
            score_improvement = ((finetuned_score - baseline_score) / baseline_score) * 100
        else:
            score_improvement = 0.0 if finetuned_score == 0 else 100.0
        
        comparison_data.append({
            'query': query,
            'baseline_avg_score': baseline_score,
            'finetuned_avg_score': finetuned_score,
            'baseline_avg_rating': baseline_result['avg_rating'],
            'finetuned_avg_rating': finetuned_result['avg_rating'],
            'score_improvement': score_improvement,
        })
    
    df = pd.DataFrame(comparison_data)
    return df


def run_demo_evaluation(
    baseline_retriever: WongnaiRetriever,
    finetuned_retriever: WongnaiRetriever,
    qa_generator: WongnaiQAGenerator,
) -> Dict:
    """Run a complete demo evaluation comparing baseline and finetuned models."""
    demo_queries = get_demo_queries()
    test_queries = [q['query'] for q in demo_queries]
    
    print("Running evaluation on demo queries...")
    print(f"Total queries: {len(test_queries)}\n")
    
    comparison_df = compare_retrievers(baseline_retriever, finetuned_retriever, test_queries)
    
    per_query_results = []
    
    print("Processing queries and generating answers...")
    print("-" * 80)
    
    for i, query_info in enumerate(demo_queries):
        query = query_info['query']
        category = query_info['category']
        
        baseline_results = baseline_retriever.search(query, top_k=5)
        finetuned_results = finetuned_retriever.search(query, top_k=5)
        
        baseline_answer = qa_generator.generate_answer(query, baseline_results)
        finetuned_answer = qa_generator.generate_answer(query, finetuned_results)
        
        row = comparison_df.iloc[i]
        
        per_query_results.append({
            'category': category,
            'query': query,
            'description': query_info['description'],
            'baseline_results': baseline_results,
            'finetuned_results': finetuned_results,
            'baseline_answer': baseline_answer,
            'finetuned_answer': finetuned_answer,
            'baseline_avg_score': row['baseline_avg_score'],
            'finetuned_avg_score': row['finetuned_avg_score'],
            'baseline_avg_rating': row['baseline_avg_rating'],
            'finetuned_avg_rating': row['finetuned_avg_rating'],
            'score_improvement': row['score_improvement'],
        })
        
        print(f"Query {i+1}/{len(demo_queries)}: {query}")
    
    print("-" * 80)
    
    category_metrics = defaultdict(lambda: {
        'count': 0,
        'baseline_avg_score': [],
        'finetuned_avg_score': [],
        'baseline_avg_rating': [],
        'finetuned_avg_rating': [],
        'score_improvements': [],
    })
    
    for result in per_query_results:
        cat = result['category']
        category_metrics[cat]['count'] += 1
        category_metrics[cat]['baseline_avg_score'].append(result['baseline_avg_score'])
        category_metrics[cat]['finetuned_avg_score'].append(result['finetuned_avg_score'])
        category_metrics[cat]['baseline_avg_rating'].append(result['baseline_avg_rating'])
        category_metrics[cat]['finetuned_avg_rating'].append(result['finetuned_avg_rating'])
        category_metrics[cat]['score_improvements'].append(result['score_improvement'])
    
    summary_metrics = {}
    for cat, metrics in category_metrics.items():
        summary_metrics[cat] = {
            'count': metrics['count'],
            'baseline_avg_score': float(np.mean(metrics['baseline_avg_score'])),
            'finetuned_avg_score': float(np.mean(metrics['finetuned_avg_score'])),
            'baseline_avg_rating': float(np.mean(metrics['baseline_avg_rating'])),
            'finetuned_avg_rating': float(np.mean(metrics['finetuned_avg_rating'])),
            'avg_improvement': float(np.mean(metrics['score_improvements'])),
        }
    
    overall = {
        'baseline_avg_score': float(comparison_df['baseline_avg_score'].mean()),
        'finetuned_avg_score': float(comparison_df['finetuned_avg_score'].mean()),
        'baseline_avg_rating': float(comparison_df['baseline_avg_rating'].mean()),
        'finetuned_avg_rating': float(comparison_df['finetuned_avg_rating'].mean()),
        'avg_improvement': float(comparison_df['score_improvement'].mean()),
    }
    
    results = {
        'queries': demo_queries,
        'comparison_df': comparison_df,
        'per_query_results': per_query_results,
        'category_summary': summary_metrics,
        'overall_metrics': overall,
    }
    
    print_evaluation_report(results)
    
    return results


def print_evaluation_report(results: Dict) -> None:
    """Print a nicely formatted evaluation report in Thai."""
    print("\n" + "=" * 80)
    print("รายงานผลการประเมินระบบ Wongnai QA".center(80))
    print("=" * 80)
    
    cat_names = {
        'cuisine': 'ประเภทอาหาร (Cuisine)',
        'food_type': 'ประเภทอาหาร/เมนู (Food Type)',
        'atmosphere_price': 'บรรยากาศและราคา (Atmosphere/Price)',
        'location': 'สถานที่ (Location)',
        'combined': 'แบบผสม (Combined)',
    }
    
    print("\n[1] ผลการประเมินแยกตามประเภทคำถาม")
    print("-" * 80)
    
    category_summary = results['category_summary']
    
    for cat_key, cat_name in cat_names.items():
        if cat_key not in category_summary:
            continue
        
        metrics = category_summary[cat_key]
        print(f"\n{cat_name} ({metrics['count']} คำถาม)")
        print(f"  - คะแนนความเกี่ยวข้องเฉลี่ย (Baseline): {metrics['baseline_avg_score']:.4f}")
        print(f"  - คะแนนความเกี่ยวข้องเฉลี่ย (Finetuned): {metrics['finetuned_avg_score']:.4f}")
        print(f"  - คะแนนดาวเฉลี่ย (Baseline): {metrics['baseline_avg_rating']:.2f}")
        print(f"  - คะแนนดาวเฉลี่ย (Finetuned): {metrics['finetuned_avg_rating']:.2f}")
        
        improvement = metrics['avg_improvement']
        if improvement > 0:
            print(f"  - การปรับปรุง: +{improvement:.1f}% (ดีขึ้น)")
        elif improvement < 0:
            print(f"  - การปรับปรุง: {improvement:.1f}% (ลดลง)")
        else:
            print(f"  - การปรับปรุง: ไม่มีการเปลี่ยนแปลง")
    
    print("\n" + "=" * 80)
    print("[2] ผลการประเมินโดยรวม")
    print("=" * 80)
    
    overall = results['overall_metrics']
    total_queries = len(results['queries'])
    
    print(f"\nจำนวนคำถามทั้งหมด: {total_queries} คำถาม")
    print(f"\nคะแนนความเกี่ยวข้องเฉลี่ย:")
    print(f"  - Baseline:  {overall['baseline_avg_score']:.4f}")
    print(f"  - Finetuned: {overall['finetuned_avg_score']:.4f}")
    
    print(f"\nคะแนนดาวเฉลี่ยของผลลัพธ์:")
    print(f"  - Baseline:  {overall['baseline_avg_rating']:.2f}")
    print(f"  - Finetuned: {overall['finetuned_avg_rating']:.2f}")
    
    print("\n" + "=" * 80)
    print("[3] สรุปผลการปรับปรุง")
    print("=" * 80)
    
    avg_improvement = overall['avg_improvement']
    
    if avg_improvement > 5:
        status = "มีการปรับปรุงอย่างมีนัยสำคัญ"
        recommendation = "แนะนำให้ใช้โมเดลที่ Finetuned"
    elif avg_improvement > 0:
        status = "มีการปรับปรุงเล็กน้อย"
        recommendation = "สามารถใช้โมเดลที่ Finetuned ได้"
    elif avg_improvement > -5:
        status = "ไม่มีการเปลี่ยนแปลงที่สำคัญ"
        recommendation = "อาจใช้โมเดล Baseline ก็ได้"
    else:
        status = "ประสิทธิภาพลดลง"
        recommendation = "ควรใช้โมเดล Baseline"
    
    print(f"\nการปรับปรุงเฉลี่ย: {avg_improvement:+.1f}%")
    print(f"สถานะ: {status}")
    print(f"คำแนะนำ: {recommendation}")
    
    print("\n" + "=" * 80)
    print("จบรายงาน".center(80))
    print("=" * 80 + "\n")

print("Evaluation module loaded successfully!")

Evaluation module loaded successfully!


## 7. Gradio Web UI

โมดูลสำหรับสร้าง Web Interface ด้วย Gradio

In [8]:
"""Gradio web UI for Wongnai QA System."""

import os
from typing import Dict, List, Optional, Tuple

import gradio as gr

# Global variables for loaded components
_system_components: Optional[Dict] = None

# Demo queries organized by category
DEMO_QUERIES = {
    "อาหารทะเล": [
        "อาหารทะเลสดๆ แถวพัทยา ราคาไม่แพง",
        "ร้านกุ้งเผาอร่อยในกรุงเทพ",
        "หอยนางรมสด ร้านไหนดี",
    ],
    "อาหารญี่ปุ่น": [
        "ซูชิราคาถูก คุณภาพดี",
        "ร้านราเมนเจ้าอร่อย",
        "บุฟเฟ่ต์อาหารญี่ปุ่น",
    ],
    "อาหารไทย": [
        "ต้มยำกุ้งรสเด็ด",
        "ร้านผัดไทยอร่อยๆ",
        "ข้าวซอยเชียงใหม่แท้ๆ",
    ],
    "บรรยากาศ": [
        "ร้านอาหารบรรยากาศดี วิวสวย",
        "คาเฟ่เงียบสงบ นั่งทำงาน",
        "ร้านอาหารสำหรับเดท",
    ],
    "ราคา": [
        "อาหารอร่อยราคาถูก",
        "บุฟเฟ่ต์ราคาหลักร้อย",
        "ร้านหรูราคาไม่แพง",
    ],
}

# Filter options
CUISINE_OPTIONS = ["", "ไทย", "ญี่ปุ่น", "จีน", "เกาหลี", "อิตาลี", "ฝรั่งเศส", "อินเดีย", "เวียดนาม"]
FOOD_OPTIONS = ["", "อาหารทะเล", "พิซซ่า", "ก๋วยเตี๋ยว", "กาแฟ", "เบเกอรี่", "สเต๊ก", "แกง", "ของหวาน"]
RATING_OPTIONS = [("ไม่กำหนด", None), ("1 ดาว", 1), ("2 ดาว", 2), ("3 ดาว", 3), ("4 ดาว", 4), ("5 ดาว", 5)]

# Custom CSS
CUSTOM_CSS = '''
@import url('https://fonts.googleapis.com/css2?family=Sarabun:wght@300;400;500;600;700&display=swap');

* {
    font-family: 'Sarabun', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
'''


def format_stars_html(rating: int) -> str:
    """Format star rating as HTML with star symbols."""
    stars = "★" * rating
    empty = "☆" * (5 - rating)
    return f'<span style="color: #ffc107; font-size: 1.1rem;">{stars}</span><span style="color: #e0e0e0;">{empty}</span>'


def format_review_card(result: Dict, rank: int) -> str:
    """Format a single review result as an HTML card."""
    star_html = format_stars_html(result.get("star_rating", 0))
    score = result.get("relevance_score", 0)
    
    tags_html = ""
    
    cuisines = result.get("cuisine_type", [])
    if cuisines:
        for c in cuisines[:3]:
            tags_html += f'<span style="display: inline-block; background: #e3f2fd; color: #1976d2; padding: 0.2rem 0.6rem; border-radius: 12px; font-size: 0.8rem; margin: 0.2rem;">{c}</span>'
    
    foods = result.get("food_type", [])
    if foods:
        for f in foods[:3]:
            tags_html += f'<span style="display: inline-block; background: #f3e5f5; color: #7b1fa2; padding: 0.2rem 0.6rem; border-radius: 12px; font-size: 0.8rem; margin: 0.2rem;">{f}</span>'
    
    locations = result.get("location", [])
    if locations:
        for l in locations[:2]:
            tags_html += f'<span style="display: inline-block; background: #e8f5e9; color: #388e3c; padding: 0.2rem 0.6rem; border-radius: 12px; font-size: 0.8rem; margin: 0.2rem;">{l}</span>'
    
    review_text = result.get("review_text", "")
    if len(review_text) > 400:
        review_text = review_text[:400] + "..."
    
    return f'''
    <div style="background: white; border-radius: 12px; padding: 1.5rem; margin: 1rem 0; box-shadow: 0 2px 8px rgba(255, 107, 53, 0.1); border: 1px solid #ffe5dc;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 0.75rem;">
            <div style="display: flex; align-items: center; gap: 0.75rem;">
                <div style="background: linear-gradient(135deg, #ff6b35, #e63946); color: white; width: 28px; height: 28px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 600; font-size: 0.9rem;">{rank}</div>
                <div>{star_html} <span style="color: #666; font-size: 0.9rem;">({result.get("star_rating", 0)}/5)</span></div>
            </div>
            <div style="background: #fff3e0; color: #ff6b35; padding: 0.25rem 0.75rem; border-radius: 20px; font-size: 0.85rem; font-weight: 500;">ความเกี่ยวข้อง: {score:.1%}</div>
        </div>
        <div style="margin-bottom: 0.75rem;">{tags_html}</div>
        <div style="color: #444; line-height: 1.6;">{review_text}</div>
    </div>
    '''


def load_system() -> Dict:
    """Load all system components."""
    global _system_components
    
    if _system_components is not None:
        return _system_components
    
    components = {
        "baseline_retriever": None,
        "finetuned_retriever": None,
        "qa_generator": None,
        "llm_generator": None,
        "loaded": False,
        "error_message": None,
    }
    
    try:
        if os.path.exists(FAISS_INDEX_PATH):
            components["baseline_retriever"] = WongnaiRetriever()
            components["baseline_retriever"].load_index()
        else:
            components["error_message"] = f"Baseline index not found at {FAISS_INDEX_PATH}. Please build the index first."
            return components
        
        if os.path.exists(FINETUNED_FAISS_INDEX_PATH):
            components["finetuned_retriever"] = WongnaiRetriever(
                model_name=MODEL_CONFIG["embedding_model"],
                index_path=FINETUNED_FAISS_INDEX_PATH,
                is_finetuned=True,
            )
            components["finetuned_retriever"].load_index()
        
        components["qa_generator"] = WongnaiQAGenerator(use_llm=False)
        
        try:
            components["llm_generator"] = WongnaiQAGenerator(use_llm=True)
        except Exception as e:
            print(f"LLM generator not available: {e}")
            components["llm_generator"] = None
        
        components["loaded"] = True
        
    except Exception as e:
        components["error_message"] = f"Error loading system: {str(e)}"
    
    _system_components = components
    return components


def search_and_answer(
    query: str,
    num_results: int,
    mode: str,
    min_rating,
    cuisine: str,
    food_type: str,
) -> Tuple[str, str]:
    """Search and generate answer based on user query."""
    components = load_system()
    
    if not components["loaded"]:
        error_html = f'<div style="background: #ffebee; color: #c62828; padding: 1rem; border-radius: 8px;">{components.get("error_message", "System not loaded")}</div>'
        return error_html, ""
    
    if not query.strip():
        return "<div style='background: #ffebee; color: #c62828; padding: 1rem;'>กรุณาป้อนคำถามหรือคำค้นหา</div>", ""
    
    cuisine_filter = cuisine if cuisine else None
    food_filter = food_type if food_type else None
    rating_filter = min_rating if min_rating else None
    
    try:
        if mode == "compare":
            return _search_compare_mode(query, num_results, components, rating_filter, cuisine_filter, food_filter)
        else:
            return _search_single_mode(query, num_results, mode, components, rating_filter, cuisine_filter, food_filter)
    except Exception as e:
        error_html = f'<div style="background: #ffebee; color: #c62828; padding: 1rem;">เกิดข้อผิดพลาด: {str(e)}</div>'
        return error_html, ""


def _search_single_mode(
    query: str,
    num_results: int,
    mode: str,
    components: Dict,
    min_rating,
    cuisine,
    food_type,
) -> Tuple[str, str]:
    """Search using a single model mode."""
    if mode == "finetuned" and components["finetuned_retriever"]:
        retriever = components["finetuned_retriever"]
    else:
        retriever = components["baseline_retriever"]
    
    results = retriever.search_with_filters(
        query=query,
        top_k=num_results,
        min_rating=min_rating,
        cuisine_filter=cuisine,
        food_type_filter=food_type,
    )
    
    if components["llm_generator"] and mode == "finetuned":
        answer = components["llm_generator"].generate_answer(query, results)
    else:
        answer = components["qa_generator"].generate_answer(query, results)
    
    answer_html = f'''
    <div style="background: linear-gradient(135deg, #fff8f5 0%, #ffffff 100%); border-left: 4px solid #ff6b35; padding: 1.5rem; border-radius: 0 12px 12px 0; margin: 1rem 0;">
        <div style="font-weight: 600; color: #ff6b35; margin-bottom: 0.5rem;">คำตอบสำหรับ: "{query}"</div>
        <div style="line-height: 1.8; color: #2d3436; white-space: pre-wrap;">{answer}</div>
    </div>
    '''
    
    results_html = ""
    if not results:
        results_html = '<div style="background: white; border-radius: 12px; padding: 1.5rem;"><p>ไม่พบผลลัพธ์ที่ตรงกับเงื่อนไข</p></div>'
    else:
        for i, result in enumerate(results, 1):
            formatted = components["qa_generator"].format_results_for_display(query, results, answer)
            results_html += format_review_card(formatted["results"][i-1], i)
    
    return answer_html, results_html


def _search_compare_mode(
    query: str,
    num_results: int,
    components: Dict,
    min_rating,
    cuisine,
    food_type,
) -> Tuple[str, str]:
    """Search using both models for comparison."""
    baseline_results = components["baseline_retriever"].search_with_filters(
        query=query, top_k=num_results, min_rating=min_rating,
        cuisine_filter=cuisine, food_type_filter=food_type,
    )
    baseline_answer = components["qa_generator"].generate_answer(query, baseline_results)
    
    if components["finetuned_retriever"]:
        finetuned_results = components["finetuned_retriever"].search_with_filters(
            query=query, top_k=num_results, min_rating=min_rating,
            cuisine_filter=cuisine, food_type_filter=food_type,
        )
        if components["llm_generator"]:
            finetuned_answer = components["llm_generator"].generate_answer(query, finetuned_results)
        else:
            finetuned_answer = components["qa_generator"].generate_answer(query, finetuned_results)
    else:
        finetuned_results = baseline_results
        finetuned_answer = "(โมเดล Fine-tuned ไม่พร้อมใช้งาน)"
    
    answer_html = f'''
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 1.5rem;">
        <div style="background: white; border-radius: 12px; padding: 1.5rem; border: 2px solid #74b9ff;">
            <h4 style="color: #1976d2; margin-bottom: 1rem;">Baseline Model</h4>
            <div style="line-height: 1.8; white-space: pre-wrap;">{baseline_answer}</div>
        </div>
        <div style="background: white; border-radius: 12px; padding: 1.5rem; border: 2px solid #00b894;">
            <h4 style="color: #388e3c; margin-bottom: 1rem;">Fine-tuned Model</h4>
            <div style="line-height: 1.8; white-space: pre-wrap;">{finetuned_answer}</div>
        </div>
    </div>
    '''
    
    results_html = "<h4 style='margin: 1.5rem 0 1rem 0; color: #ff6b35;'>ผลลัพธ์จาก Baseline Model:</h4>"
    if baseline_results:
        formatted = components["qa_generator"].format_results_for_display(query, baseline_results, baseline_answer)
        for i, result in enumerate(formatted["results"], 1):
            results_html += format_review_card(result, i)
    else:
        results_html += '<div style="background: white; border-radius: 12px; padding: 1.5rem;"><p>ไม่พบผลลัพธ์</p></div>'
    
    if components["finetuned_retriever"]:
        results_html += "<h4 style='margin: 1.5rem 0 1rem 0; color: #ff6b35;'>ผลลัพธ์จาก Fine-tuned Model:</h4>"
        if finetuned_results:
            formatted = components["qa_generator"].format_results_for_display(query, finetuned_results, finetuned_answer)
            for i, result in enumerate(formatted["results"], 1):
                results_html += format_review_card(result, i)
        else:
            results_html += '<div style="background: white; border-radius: 12px; padding: 1.5rem;"><p>ไม่พบผลลัพธ์</p></div>'
    
    return answer_html, results_html


def launch_app(share: bool = False) -> None:
    """Build and launch the Gradio web interface."""
    components = load_system()
    
    theme = gr.themes.Soft(
        primary_hue="orange",
        secondary_hue="red",
        neutral_hue="stone",
    ).set(
        body_background_fill="linear-gradient(135deg, #fff8f5 0%, #fff0eb 100%)",
        button_primary_background_fill="linear-gradient(135deg, #ff6b35 0%, #e63946 100%)",
        button_primary_background_fill_hover="linear-gradient(135deg, #e55a2b 0%, #d32f3f 100%)",
        button_primary_text_color="white",
    )
    
    with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Wongnai QA System") as demo:
        gr.Markdown('''
        <h1 style="text-align: center; background: linear-gradient(135deg, #ff6b35 0%, #e63946 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Wongnai QA System</h1>
        <p style="text-align: center; color: #636e72; font-size: 1.1rem;">ระบบถาม-ตอบร้านอาหารอัจฉริยะ ใช้ AI ช่วยค้นหาและแนะนำร้านอาหารจากรีวิวจริง</p>
        ''')
        
        if not components["loaded"]:
            gr.Markdown(f'''
            <div style="background: #ffebee; color: #c62828; padding: 1rem; border-radius: 8px;">
                ⚠️ {components.get("error_message", "ระบบยังไม่พร้อมใช้งาน")}<br>
                กรุณาสร้าง FAISS index ก่อนใช้งาน
            </div>
            ''')
        
        with gr.Tab("ค้นหา"):
            with gr.Row():
                with gr.Column(scale=2):
                    query_input = gr.Textbox(
                        label="ถามเกี่ยวกับร้านอาหาร",
                        placeholder="เช่น อาหารทะเลสดๆ แถวพัทยา ราคาไม่แพง",
                        lines=2,
                    )
                    
                    with gr.Row():
                        num_results = gr.Slider(
                            label="จำนวนผลลัพธ์", minimum=1, maximum=10, value=5, step=1,
                        )
                        model_mode = gr.Radio(
                            label="โหมดการค้นหา",
                            choices=[
                                ("Baseline Model", "baseline"),
                                ("Fine-tuned Model", "finetuned"),
                                ("เปรียบเทียบทั้งสอง", "compare"),
                            ],
                            value="baseline",
                        )
                    
                    with gr.Accordion("ตัวกรองเพิ่มเติม", open=False):
                        with gr.Row():
                            min_rating = gr.Dropdown(
                                label="คะแนนขั้นต่ำ", choices=RATING_OPTIONS, value=None,
                            )
                            cuisine_filter = gr.Dropdown(
                                label="ประเภทอาหาร", choices=CUISINE_OPTIONS, value="",
                            )
                            food_filter = gr.Dropdown(
                                label="แนวอาหาร", choices=FOOD_OPTIONS, value="",
                            )
                    
                    search_btn = gr.Button("🔍 ค้นหา", variant="primary", size="lg")
                
                with gr.Column(scale=3):
                    answer_output = gr.HTML(label="คำตอบ")
                    with gr.Accordion("ดูรีวิวที่เกี่ยวข้อง", open=True):
                        results_output = gr.HTML()
        
        with gr.Tab("ตัวอย่างคำถาม"):
            gr.Markdown("### เลือกคำถามตัวอย่างเพื่อทดสอบระบบ")
            
            demo_outputs = [query_input, num_results, model_mode, cuisine_filter, food_filter]
            
            for category, queries in DEMO_QUERIES.items():
                with gr.Row():
                    gr.Markdown(f"**{category}**")
                with gr.Row():
                    for query in queries:
                        btn = gr.Button(query, size="sm", variant="secondary")
                        btn.click(
                            fn=lambda q=query: (q, 5, "compare", "", ""),
                            inputs=[],
                            outputs=demo_outputs,
                        )
        
        with gr.Tab("เกี่ยวกับระบบ"):
            gr.Markdown('''
            <div style="background: white; border-radius: 12px; padding: 2rem; margin: 1rem 0; border: 1px solid #ffe5dc;">
                <h3 style="color: #ff6b35;">Wongnai QA System - ระบบถาม-ตอบร้านอาหารอัจฉริยะ</h3>
                <p style="line-height: 1.8;">
                    ระบบถาม-ตอบร้านอาหารที่ใช้เทคโนโลยี Natural Language Processing (NLP) 
                    และ Large Language Models (LLM) เพื่อช่วยค้นหาและแนะนำร้านอาหารจากรีวิวจริงบน Wongnai
                </p>
            </div>
            ''')
            
            gr.Markdown('''
            <div style="background: white; border-radius: 12px; padding: 2rem; margin: 1rem 0; border: 1px solid #ffe5dc;">
                <h3 style="color: #ff6b35;">ข้อมูลโมเดล</h3>
                <ul style="line-height: 2;">
                    <li><strong>Embedding Model:</strong> intfloat/multilingual-e5-base</li>
                    <li><strong>QA Model:</strong> scb10x/llama3.1-typhoon2-8b-instruct (Thai LLM)</li>
                    <li><strong>Vector Database:</strong> FAISS</li>
                    <li><strong>Framework:</strong> Hugging Face Transformers</li>
                </ul>
            </div>
            ''')
        
        search_btn.click(
            fn=search_and_answer,
            inputs=[query_input, num_results, model_mode, min_rating, cuisine_filter, food_filter],
            outputs=[answer_output, results_output],
        )
    
    demo.launch(server_name="0.0.0.0", server_port=7860, share=share, show_error=True)

print("Web UI module loaded successfully!")

Web UI module loaded successfully!


## 8. Main Pipeline

รัน Pipeline ทั้งหมด

In [9]:
# ==========================================
# STEP 1: Data Preprocessing
# ==========================================

def run_preprocessing():
    """Run data preprocessing pipeline."""
    print("=" * 60)
    print("Wongnai QA System - Data Preprocessing Pipeline")
    print("=" * 60)
    
    print("\n[1/4] Loading reviews...")
    reviews_df = load_reviews(REVIEW_TRAIN_FILE)
    print(f"Loaded {len(reviews_df)} reviews")
    
    print("\n[2/4] Loading food dictionary...")
    food_dict = load_food_dictionary(FOOD_DICT_FILE)
    food_dict = [f for f in food_dict if 4 <= len(f) <= 50]
    food_dict = list(dict.fromkeys(food_dict))[:5000]
    food_dict_set = set(food_dict)
    print(f"Loaded {len(food_dict)} food items")
    
    print("\n[3/4] Loading queries...")
    queries_judges = load_queries(QUERY_JUDGES_FILE)
    queries_algo = load_queries(QUERY_ALGO_FILE)
    print(f"Loaded {len(queries_judges)} judge-labeled queries")
    print(f"Loaded {len(queries_algo)} algorithm-labeled queries")
    
    print("\n[4/4] Processing reviews...")
    processed_df = process_all_reviews(reviews_df, food_dict_set)
    
    print("\n" + "=" * 60)
    print("STATISTICS")
    print("=" * 60)
    
    print(f"\nTotal reviews processed: {len(processed_df)}")
    
    print("\nRating distribution:")
    rating_dist = processed_df['star_rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        print(f"  {rating} stars: {count} ({count/len(processed_df)*100:.1f}%)")
    
    print("\n" + "=" * 60)
    print("Preprocessing completed successfully!")
    print("=" * 60)
    
    return processed_df, food_dict, queries_judges, queries_algo


# Uncomment to run preprocessing
# processed_df, food_dict, queries_judges, queries_algo = run_preprocessing()

In [10]:
# ==========================================
# STEP 2: Build FAISS Index (Baseline)
# ==========================================

def build_baseline():
    """Build baseline FAISS index."""
    print("=" * 60)
    print("Building Baseline FAISS Index")
    print("=" * 60)
    
    # Load processed data
    processed_file = os.path.join(PROCESSED_DATA_PATH, "processed_reviews.pkl")
    if not os.path.exists(processed_file):
        print(f"Error: Processed data not found at {processed_file}")
        print("Please run preprocessing first.")
        return None
    
    processed_df = pd.read_pickle(processed_file)
    print(f"Loaded {len(processed_df)} processed reviews\n")
    
    # Build index
    retriever = build_baseline_index(processed_df)
    
    print(f"\nIndex Statistics")
    print("-" * 60)
    print(f"Index path: {FAISS_INDEX_PATH}")
    print(f"Number of vectors: {retriever.index.ntotal}")
    print(f"Vector dimension: {retriever.embeddings.shape[1]}")
    print(f"Model used: {retriever.model_name}")
    print("\n✓ Baseline index built successfully!")
    
    return retriever


# Uncomment to build baseline index
# baseline_retriever = build_baseline()

In [11]:
# ==========================================
# STEP 3: Fine-tune Embedding Model
# ==========================================

def run_finetuning():
    """Run fine-tuning pipeline."""
    print("=" * 60)
    print("Fine-tuning Embedding Model")
    print("=" * 60)
    
    # Load processed data
    processed_file = os.path.join(PROCESSED_DATA_PATH, "processed_reviews.pkl")
    if not os.path.exists(processed_file):
        print(f"Error: Processed data not found at {processed_file}")
        return None
    
    processed_df = pd.read_pickle(processed_file)
    print(f"[1/4] Loaded {len(processed_df)} processed reviews")
    
    # Load food dictionary and queries
    print("\n[2/4] Loading food dictionary and queries...")
    food_dict = load_food_dictionary(FOOD_DICT_FILE)
    queries_judges = load_queries(QUERY_JUDGES_FILE)
    queries_algo = load_queries(QUERY_ALGO_FILE)
    print(f"  Loaded {len(food_dict)} food items")
    print(f"  Loaded {len(queries_judges)} judge-labeled queries")
    print(f"  Loaded {len(queries_algo)} algorithm-labeled queries")
    
    # Generate training pairs
    print("\n[3/4] Generating training pairs...")
    training_pairs = generate_training_pairs(
        processed_df=processed_df,
        food_dict=food_dict,
        queries_judges=queries_judges,
        queries_algo=queries_algo,
        num_pairs=10000
    )
    
    # Fine-tune model
    print("\n[4/4] Fine-tuning model...")
    finetuned_path = finetune_model(
        training_pairs=training_pairs,
        output_path='models/finetuned_embedding',
        epochs=3,
        batch_size=16,
        warmup_steps=100
    )
    
    # Build finetuned index
    print("\n[5/5] Building finetuned FAISS index...")
    retriever = build_finetuned_index(processed_df, finetuned_path)
    
    print("\n" + "=" * 60)
    print("Training Summary")
    print("=" * 60)
    print(f"Base model: {MODEL_CONFIG['embedding_model']}")
    print(f"Training pairs: {len(training_pairs)}")
    print(f"Epochs: 3")
    print(f"Batch size: 16")
    print(f"Model saved to: {finetuned_path}")
    print(f"Index saved to: {FINETUNED_FAISS_INDEX_PATH}")
    print(f"Number of vectors in index: {retriever.index.ntotal}")
    print("\n✓ Fine-tuning completed successfully!")
    
    return retriever


# Uncomment to run fine-tuning
# finetuned_retriever = run_finetuning()

In [12]:
# ==========================================
# STEP 4: Run Evaluation
# ==========================================

def run_evaluation():
    """Run evaluation comparing baseline and finetuned models."""
    print("=" * 60)
    print("Evaluating Retrievers")
    print("=" * 60)
    
    # Load baseline retriever
    print("[1/3] Loading baseline retriever...")
    baseline_retriever = WongnaiRetriever()
    baseline_retriever.load_index()
    
    # Load finetuned retriever if available
    finetuned_retriever = None
    if os.path.exists(FINETUNED_FAISS_INDEX_PATH):
        print("\n[2/3] Loading finetuned retriever...")
        finetuned_retriever = WongnaiRetriever(
            model_name=MODEL_CONFIG['embedding_model'],
            index_path=FINETUNED_FAISS_INDEX_PATH,
            is_finetuned=True,
        )
        finetuned_retriever.load_index()
    else:
        print("\n[2/3] Finetuned index not found, using baseline for comparison...")
        finetuned_retriever = baseline_retriever
    
    # Initialize QA generator
    print("\n[3/3] Initializing QA generator...")
    qa_generator = WongnaiQAGenerator(use_llm=False)
    
    # Run evaluation
    print("\n" + "=" * 70)
    print("Running evaluation on demo queries...")
    print("=" * 70)
    
    results = run_demo_evaluation(
        baseline_retriever=baseline_retriever,
        finetuned_retriever=finetuned_retriever,
        qa_generator=qa_generator
    )
    
    print("\n✓ Evaluation completed successfully!")
    return results


# Uncomment to run evaluation
# evaluation_results = run_evaluation()

In [ ]:
# ==========================================
# STEP 5: Launch Web UI
# ==========================================

# Launch Gradio Web UI
# Uncomment to run the web interface
launch_app(share=False)

## Quick Demo

ทดลองค้นหาด้วยคำถามตัวอย่าง

In [14]:
# Quick demo with sample queries

def demo_search():
    """Run a quick demo search."""
    # Load retriever
    retriever = WongnaiRetriever()
    retriever.load_index()
    
    # Initialize QA generator
    qa_generator = WongnaiQAGenerator(use_llm=False)
    
    # Sample queries
    demo_queries = [
        "อาหารญี่ปุ่นอร่อย",
        "ร้านอาหารทะเลสดๆ",
        "คาเฟ่บรรยากาศดีกรุงเทพ",
        "ร้านอาหารหรูหรา",
    ]
    
    print("=" * 70)
    print("DEMO MODE - SAMPLE QUERIES")
    print("=" * 70)
    
    for i, query in enumerate(demo_queries, 1):
        print(f"\n{'='*70}")
        print(f"Query {i}/{len(demo_queries)}: {query}")
        print(f"{'='*70}")
        
        results = retriever.search(query, top_k=3)
        answer = qa_generator.generate_answer(query, results)
        
        print(f"\nAnswer:")
        print(answer[:500] + "..." if len(answer) > 500 else answer)
    
    print("\n" + "=" * 70)
    print("Demo completed!")
    print("=" * 70)


# Uncomment to run demo
# demo_search()

## สรุป

Notebook นี้รวมทุกโมดูลของ Wongnai QA System:

1. **Config** - กำหนดค่าคอนฟิกและ path
2. **Data Preprocessing** - โหลดและประมวลผลข้อมูลรีวิว
3. **Retrieval** - สร้าง FAISS index และค้นหา
4. **Fine-tuning** - Fine-tune embedding model
5. **QA Generator** - สร้างคำตอบจากรีวิว
6. **Evaluation** - ประเมินผลระบบ
7. **Web UI** - Gradio interface
8. **Pipeline** - รัน pipeline ทั้งหมด

### วิธีใช้งาน
1. รัน cell ตั้งแต่ต้นเพื่อโหลดทุกโมดูล
2. รัน preprocessing เพื่อประมวลผลข้อมูล
3. รัน build baseline index
4. (Optional) รัน fine-tuning
5. รัน evaluation หรือ launch web UI